# Corrective Retrieval Augmented Generation: a Windows-native course

Shi-Qi Yan, Jia-Chen Gu, Yun Zhu, Zhen-Hua Ling, [*Corrective Retrieval Augmented Generation*, arXiv:2401.15884](https://arxiv.org/abs/2401.15884).

This notebook follows the paper's evaluate → route → refine → generate structure. It adapts that flow for a small Windows course. Run cells in order with **Restart Kernel and Run All**. The first run may download HotpotQA and make provider-metered Agnes calls; later runs reuse versioned caches. The course uses no search key, Docker, WSL, paid embedding API, or LangChain.

## 1. Why naive RAG fails

A retriever ranks similarity. It does not decide whether the evidence is sufficient. A passage can share every query word while describing the wrong person, and a multi-hop question can retrieve one useful bridge while missing the other. Generating directly from those passages can produce a plausible, unsupported answer. Correction adds a decision between retrieval and generation, though the evaluator can still admit distractors or discard useful facts.

Ask three separate questions: did retrieval find the annotated evidence? Did refinement keep the necessary facts? Does the answer follow from those facts? High relevance does not establish a complete multi-hop chain, and a literal substring match does not establish factual correctness.

## 2. CRAG paper map

| Paper component | This course | Important difference |
|---|---|---|
| Retrieval evaluator (§4.2) | Agnes JSON score for each passage | Paper uses a trained T5-large (0.77B) evaluator; we do not train it |
| Correct / Incorrect / Ambiguous (§4.3) | Maximum score with upper .7 and lower .3 | Heuristic 0–1 thresholds, not calibrated paper defaults |
| Knowledge refinement (§4.4) | Select exact sentence IDs | A prompted approximation of fine-grained filtering |
| Knowledge searching (§4.5) | Optional real web snippets | Disabled in reported runs; Incorrect therefore abstains |
| Generator | Agnes Chat Completions | Fixed agnes-3.0-flash, not the paper's generator setup |

The paper evaluates PopQA, Biography, PubHealth and ARC. This course uses HotpotQA for inspectable multi-hop evidence. Neither its data nor its scores reproduce paper results. The paper's relevance scale is not our 0–1 prompt scale.

## 3. Environment check

Store credentials in Windows **user environment variables**. Restart the terminal or Jupyter launcher after changing them so the process inherits them. `.env.example` lists variable names only and is not loaded. `AGNES_BASE_URL` is informational; the implementation uses the required fixed endpoint.

### Check imports and credential presence

**Motivation.** Fail before downloading data or making API requests if setup is incomplete.

**Paper mapping.** Infrastructure used by every stage.

**Next cell.** Locate the repo from either its root or notebooks directory and print booleans only.

**Failure signals.** A missing variable, Python below 3.11, or ModuleNotFoundError means setup must be corrected first.

**Read the output.** Both credential flags should be True. No credential value is displayed.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert sys.version_info >= (3, 11)
from src.config import check_environment, MODEL_NAME, BASE_URL
env = check_environment()
print(env)
assert all(env.values()), "Set AGNESAI_API_KEY and HF_TOKEN in the Windows user environment; restart Jupyter."
print("Model:", MODEL_NAME, "Endpoint:", BASE_URL)

{'AGNESAI_API_KEY': True, 'HF_TOKEN': True}
Model: agnes-3.0-flash Endpoint: https://apihub.agnes-ai.com/v1


## 4. Dataset

[HotpotQA on Hugging Face](https://huggingface.co/datasets/hotpotqa/hotpot_qa), configuration `distractor`, split `validation`, supplies annotated supporting facts. The usual candidate pool has two gold paragraphs plus distractors; inspect actual counts instead of assuming ten for every row. A paragraph is gold when its title occurs in `supporting_facts.title`. These labels are for evaluation only and never enter model prompts.

Shuffle validation with seed 42 and select 200 rows. The smoke set takes up to four bridge and four comparison questions **from this slice**, filling from remaining slice rows if necessary. A bridge question follows an entity relation; a comparison question needs facts about both subjects. Gold titles annotate evidence, not a relevance probability.

This is not Meta Comprehensive RAG Benchmark, which shares the CRAG acronym. The dataset slice, raw download, provenance manifest, model judgements, and resumable results live under ignored `data/cache/`.

### Load the reproducible slice

**Motivation.** Make every later comparison use the same questions and labels.

**Paper mapping.** A tutorial dataset substitution, not a paper benchmark.

**Next cell.** Load or build the 200-row slice and show type and pool-size counts.

**Failure signals.** HF authentication/network errors or a wrong config prevent loading; do not fabricate rows.

**Read the output.** Expect 200 unique IDs and eight smoke IDs inside the slice; pool sizes may vary.

In [2]:
import pandas as pd
from src.data_hotpot import build_slice
records, smoke_ids = build_slice(n=200, seed=42)
records_by_id = {r["id"]: r for r in records}
assert len(records_by_id) == 200 and len(smoke_ids) == 8
assert set(smoke_ids) <= records_by_id.keys()
display(pd.Series([r["question_type"] for r in records]).value_counts().rename("questions"))
display(pd.Series([len(r["context_paragraphs"]) for r in records]).value_counts().rename("pool sizes"))
print("Smoke IDs:", smoke_ids)

bridge        158
comparison     42
Name: questions, dtype: int64

10    199
2       1
Name: pool sizes, dtype: int64

Smoke IDs: ['5add1d575542992c1e3a2540', '5ac55ea55542993e66e82377', '5ac3a76e554299741d48a2be', '5a7133565542994082a3e65c', '5ac15de55542991316484afb', '5a8cea40554299441c6b9f76', '5a8ce15d554299653c1aa12f', '5a7c634c55429907fabeef79']


### Read one complete example

**Motivation.** See the evidence before interpreting retrieval scores.

**Paper mapping.** Connects retrieval inputs to supporting-fact supervision used only for diagnostics here.

**Next cell.** Print the full first record, including every paragraph and gold title.

**Failure signals.** Missing sentences or gold labels would limit evaluation and must not be silently invented.

**Read the output.** Follow the supporting titles and ask whether both hops needed for the answer are explicit.

In [3]:
import json
print(json.dumps(records[0], indent=2, ensure_ascii=False))

{
  "id": "5add1d575542992c1e3a2540",
  "question": "What nationality was Oliver Reed's character in the film Royal Flash?",
  "gold_answer": "Prussian",
  "gold_titles": [
    "Royal Flash (film)",
    "Otto von Bismarck"
  ],
  "question_type": "bridge",
  "level": "hard",
  "context_paragraphs": [
    {
      "title": "Robin Barton",
      "text": "Robin Barton. Robin Barton (born 5 November 1958) is a British art dealer dealing primarily with Banksy's.  Barton studied photography and graphic design at the Exeter College of Art and Design and this was his first encounter with Russell Young.  Moving to London in 1980 he began working as a freelance photographer for music and fashion publications \"Sounds\", \"NME\", \"Blitz\", \"The Face\" moving on to working regularly for pioneering \"Independent Magazine\" photographing amongst others Sir Alec Guinness, Oliver Reed, Johnny Depp, Lou Reed, Hugh Grant and Sir Peter Hall.  Laterly he worked for other publications \"Sunday Times\", \"

## 5. Retriever: embedded Qdrant

TF-IDF weights lexical features. TruncatedSVD projects the sparse matrix to at most 256 latent dimensions, pads when needed, and normalizes vectors for cosine search. This free CPU baseline differs from a modern semantic embedding model. It fits candidate paragraph text only, never answers or gold labels. Unknown query vocabulary can produce weak or zero vectors.

Main retrieval filters `question_id`: each question competes only with its own supplied distractor pool. An unfiltered search over the 200-question corpus is a separate demonstration, not the reported evaluation protocol. We store labels in payloads for diagnostics but send only title and text to the evaluator. Count **and corpus/encoder fingerprints** must match before index reuse; count alone misses changed documents.

### Index and compare search scopes

**Motivation.** Keep evaluation boundaries explicit and make reruns idempotent.

**Paper mapping.** Provides the retrieved document set evaluated by CRAG.

**Next cell.** Index the slice, run pool-filtered retrieval and then an unfiltered demo.

**Failure signals.** A Qdrant lock means another kernel owns the same path; stale encoder metadata triggers a rebuild.

**Read the output.** Filtered payload IDs must all match the requested question; similarity scores are not evaluator confidence.

In [4]:
from src.qdrant_store import index_slice, close_qdrant_client
from src.retrieve import search
count = index_slice(records)
sample = records[0]
pool_hits = search(sample["question"], k=3, question_id=sample["id"])
assert all(h["payload"]["question_id"] == sample["id"] for h in pool_hits)
print("Indexed paragraphs:", count)
display(pd.DataFrame([{ "title": h["title"], "similarity": h["score"], "is_gold": h["payload"]["is_gold"]} for h in pool_hits]))
open_hits = search(sample["question"], k=5)
display(pd.DataFrame([{ "title": h["title"], "question_id": h["payload"]["question_id"]} for h in open_hits]))

[*] Collection 'hotpot_slice' already indexed with 1992 points. Skipping upsert.


Indexed paragraphs: 1992


,title,similarity,is_gold
0,Royal Flash (film),0.893053,True
1,Royal Flash,0.812065,False
2,Harry Flashman,0.801413,False


,title,question_id
0,Royal Flash (film),5add1d575542992c1e3a2540
1,Royal Flash,5add1d575542992c1e3a2540
2,Harry Flashman,5add1d575542992c1e3a2540
3,Ivan Dragomiloff,5add1d575542992c1e3a2540
4,The Duke of Hamilton,5add1d575542992c1e3a2540


## 6. Evaluator

For each passage, Agnes returns `score`, `label`, and `why`. An intermediate hop can be relevant; a shared name alone is not enough. A 0.9 score is a prompted judgement, not a calibrated 90% probability. JSON extraction accepts fences, preamble, and braces inside strings, then validates finite scores in [0,1], allowed labels, and a nonempty rationale. Transport and schema failures raise instead of becoming cached zero scores.

### Inspect document judgements

**Motivation.** A routing decision is only as good as the evidence assessment behind it.

**Paper mapping.** Approximates the retrieval evaluator in §4.2 with an LLM rather than trained T5-large.

**Next cell.** Evaluate three retrieved passages, showing each score, label and explanation.

**Failure signals.** Malformed JSON or invalid scores stop the cell; rerun after correction, reusing successful cached judgements.

**Read the output.** Compare rationales with actual text and labels; do not assume a confident explanation is correct.

In [5]:
from src.crag import evaluate_documents, decide_action, refine_strips, generate_answer
evaluations = evaluate_documents(sample["question"], pool_hits)
display(pd.DataFrame([{k: e[k] for k in ("title", "score", "label", "why")} for e in evaluations]))

,title,score,label,why
0,Royal Flash (film),1.0,relevant,The evidence confirms Oliver Reed played Otto ...
1,Royal Flash,0.0,irrelevant,The evidence identifies the film and its sourc...
2,Harry Flashman,0.2,irrelevant,The evidence identifies the actor as Malcolm M...


## 7. Actions

Let m be the maximum passage score. `m >= .7` selects Correct; `m <= .3` selects Incorrect; otherwise the route is Ambiguous. No passages also gives Incorrect. These inclusive boundaries are deliberate. Correct says that at least one passage looks strong. It does not establish that every required hop is present. Incorrect discards local evidence and abstains when web recovery is disabled. Ambiguous retains passages above the lower threshold; optional web search can supplement them. Labels explain the decision, while numeric scores control routing and filtering.

### Check routing boundaries

**Motivation.** Make the decision rule testable independently of stochastic model outputs.

**Paper mapping.** Implements the three-way control decision from §4.3 with tutorial thresholds.

**Next cell.** Show the current route and deterministic boundary examples.

**Failure signals.** Out-of-range or NaN scores and reversed thresholds must raise rather than silently route.

**Read the output.** Synthetic boundary scores are unit examples, not measured model results.

In [6]:
print("Observed action:", decide_action(evaluations))
for scores in ([], [0.3], [0.5], [0.7], [0.1, 0.8]):
    print(scores, "->", decide_action(scores))

Observed action: Correct
[] -> Incorrect
[0.3] -> Incorrect
[0.5] -> Ambiguous
[0.7] -> Correct
[0.1, 0.8] -> Correct


## 8. Strip refinement

Long paragraphs can contain distractors even when their title is useful. Split them into sentences, number those sentences, and ask the model for IDs. The selected text comes from the original sentences, so the model cannot insert a new fact through an invented strip. Exact copying preserves provenance, though it cannot guarantee relevance or enough context. The punctuation-based splitter is deliberately simple and can mishandle abbreviations.

### Keep source-backed sentences

**Motivation.** Reduce irrelevant context while retaining intermediate evidence.

**Paper mapping.** A sentence-selection approximation of knowledge refinement in §4.4.

**Next cell.** Refine one passage and print the exact retained strings.

**Failure signals.** Invalid sentence IDs fail validation; an empty result is legitimate and must not be replaced with heuristic evidence.

**Read the output.** Check whether each kept sentence answers a needed subquestion; missing a bridge can break generation.

In [7]:
kept = refine_strips(sample["question"], pool_hits[0])
print("Kept strips:", json.dumps(kept, indent=2, ensure_ascii=False))

Kept strips: [
  "Additionally, Oliver Reed appeared in the role of Otto von Bismarck, Alan Bates as Rudi von Sternberg, and Florinda Bolkan played Lola Montez."
]


## 9. Generation and three diagnostic scenarios

The generator sees only the question and selected strips. Empty evidence returns a deterministic abstention without an API request. With nonempty evidence, the model is told to abstain unless the strips establish the complete answer. That instruction does not formally guarantee grounding. Gold answers stay outside prompts.

Annotations select examples for diagnostic cases only. The ordinary pipeline retrieves and evaluates without gold labels. The hidden-gold case removes annotated paragraphs to test the evaluator. Removing gold does not mathematically force an LLM to choose Incorrect. A Correct decision in that case is an evaluator failure to inspect, not a reason to overwrite its scores.

### Generate from the inspected evidence

**Motivation.** Separate source selection from answer formulation.

**Paper mapping.** The final answer-generation stage consumes refined knowledge.

**Next cell.** Generate from the selected strips and compare with the gold answer, then demonstrate empty-evidence abstention.

**Failure signals.** A plausible answer without supporting facts is still a failure; provider failures are raised and not cached as answers.

**Read the output.** Treat answer-versus-gold as a diagnostic, not evidence of paper-level accuracy.

In [8]:
print("Answer:", generate_answer(sample["question"], kept))
print("Gold:", sample["gold_answer"])
print("Empty evidence:", generate_answer(sample["question"], []))

Answer: I don't have enough information in the provided evidence to answer this question.
Gold: Prussian
Empty evidence: I don't have enough information in the provided evidence to answer this question.


### Run three evidence stress tests

**Motivation.** Expose easy retrieval, multi-hop completeness, and evaluator false positives.

**Paper mapping.** Exercises evaluation, action selection, refinement and generation together.

**Next cell.** Find two different questions with both gold titles retrievable, then run a third question with gold paragraphs hidden.

**Failure signals.** If no suitable question exists the assertions fail transparently; a hidden-gold Correct action is a finding, not an exception.

**Read the output.** Read action, scores, retained strips and answer side by side. Diagnostic gold labels never choose strips inside run_crag.

In [9]:
from src.pipeline import run_crag
easy = next(r for r in records if set(r["gold_titles"]) <= {h["title"] for h in search(r["question"], 3, r["id"])})
multi = next(r for r in records if r["id"] != easy["id"] and r["question_type"] == "bridge" and set(r["gold_titles"]) <= {h["title"] for h in search(r["question"], 4, r["id"])})
hidden = next(r for r in records if r["id"] not in (easy["id"], multi["id"]) and len(r["context_paragraphs"]) > 2)
distractors = [{"title": p["title"], "text": p["text"], "payload": {"question_id": hidden["id"]}} for p in hidden["context_paragraphs"] if not p["is_gold"]][:3]
scenarios = [("Easy retrieval", easy, 3, None), ("Both hops retrievable", multi, 4, None), ("Gold hidden", hidden, 3, distractors)]
for name, row, k, docs in scenarios:
    result = run_crag(row["question"], row["id"], k=k, docs=docs)
    print("\n", name, row["id"], row["question"])
    print("Action:", result.action, "Actual requests:", result.n_llm_calls)
    display(pd.DataFrame([{key: e[key] for key in ("title", "score", "label", "why")} for e in result.evaluations]))
    print("Kept strips:", json.dumps(result.strips, ensure_ascii=False, indent=2))
    print("Answer:", result.answer, "\nGold:", row["gold_answer"])


 Easy retrieval 5ae27b535542996483e649b6 The chorus of "On the Radio" contains references to the song "November Rain" whose band's lead singer is?
Action: Correct Actual requests: 0


,title,score,label,why
0,On the Radio (Regina Spektor song),0.8,relevant,The evidence explicitly states that the chorus...
1,November Rain,1.0,relevant,The evidence explicitly identifies Axl Rose as...
2,Till I Die (Machine Gun Kelly song),0.0,irrelevant,The evidence discusses Machine Gun Kelly's son...


Kept strips: [
  "On the Radio (Regina Spektor song).",
  "The chorus contains references to the song \"November Rain\" by Guns N' Roses.",
  "\"November Rain\" is a power ballad by the American hard rock band Guns N' Roses.",
  "Written by the band's lead singer Axl Rose, the song was released as a single in 1992 from their third studio album, \"Use Your Illusion I\" (1991)."
]
Answer: Axl Rose 
Gold: Axl Rose

 Both hops retrievable 5a7133565542994082a3e65c What Kentucky county has a population of 60,316 and features the Lake Louisvilla neighborhood?
Action: Correct Actual requests: 0


,title,score,label,why
0,"Lake Louisvilla, Louisville",0.8,relevant,The evidence confirms the location of the Lake...
1,"Kentucky County, Virginia",0.0,irrelevant,The question asks for a county in the state of...
2,"Casey County, Kentucky",0.0,irrelevant,"The evidence describes Casey County, Kentucky,..."
3,"Oldham County, Kentucky",1.0,relevant,The evidence explicitly states that Oldham Cou...


Kept strips: [
  "Lake Louisvilla is a neighborhood partially located in Louisville, Kentucky.",
  "It is located between Westport Road in Louisville and KY 22 in Oldham County.",
  "Oldham County, Kentucky.",
  "As of the 2010 census, the population was 60,316.",
  "The county is named for Colonel William Oldham."
]
Answer: Oldham County 
Gold: Oldham County

 Gold hidden 5add1d575542992c1e3a2540 What nationality was Oliver Reed's character in the film Royal Flash?
Action: Incorrect Actual requests: 0


,title,score,label,why
0,Robin Barton,0.0,irrelevant,The evidence discusses Robin Barton's photogra...
1,Funny Bones,0.0,irrelevant,The evidence describes the film 'Funny Bones' ...
2,Ivan Dragomiloff,0.2,irrelevant,The evidence identifies the character Ivan Dra...


Kept strips: []
Answer: I don't have enough information in the provided evidence to answer this question. 
Gold: Prussian


## 10. Full pipeline on the eight-ID smoke set

Use k=3 and `allow_web=False`. Each successful ID is atomically checkpointed; rerunning skips completed IDs and retries only failures. Provider 429 responses receive bounded exponential backoff; benchmark-level retries reuse already-cached successful primitive calls. Exhausted retries raise with a sanitized stack checkpoint instead of silently dropping a question.

`gold_in_kept_titles` means at least one annotated title contributed an actual retained strip. It does not mean both gold titles survived. `n_llm_calls` counts actual HTTP attempts during this invocation, including retries; cache hits and empty-evidence abstentions count zero. Literal `answer_contains_gold` tests a nonempty case-insensitive gold substring in the answer in one direction only.

### Run and interpret smoke results

**Motivation.** Check all stages before spending requests on 50 questions.

**Paper mapping.** An engineering smoke test, not a paper evaluation protocol.

**Next cell.** Execute eight IDs, save smoke_crag.json, and derive two prose interpretations from the actual rows.

**Failure signals.** A crash leaves completed IDs checkpointed; fix the cause and rerun this cell. Never replace failed IDs with invented outputs.

**Read the output.** A substring hit can be accidental; no-hit can be a valid paraphrase. Prose below reports observations without claiming correctness.

In [10]:
from src.pipeline import run_smoke_benchmark
from IPython.display import Markdown, display
smoke_rows = run_smoke_benchmark(smoke_ids, records_by_id, k=3, allow_web=False)
smoke_df = pd.DataFrame(smoke_rows)
display(smoke_df[["id", "type", "action", "gold_in_kept_titles", "answer_contains_gold", "n_llm_calls", "cache_hit"]])
for row in smoke_rows[:2]:
    display(Markdown(f"For `{row['id']}`, the route was **{row['action']}**. At least one annotated title contributed a kept strip: **{row['gold_in_kept_titles']}**. The literal gold substring test returned **{row['answer_contains_gold']}**. The answer was `{row['generated_answer']}` versus reference `{row['gold_answer']}`. Inspect the saved strips before interpreting this as success; this run made {row['n_llm_calls']} model requests for the row."))

smoke_crag_v2: 1/8; cached=True; requests=0


smoke_crag_v2: 2/8; cached=True; requests=0


smoke_crag_v2: 3/8; cached=True; requests=0


smoke_crag_v2: 4/8; cached=True; requests=0


smoke_crag_v2: 5/8; cached=True; requests=0


...

smoke_crag_v2: 7/8; cached=True; requests=0


smoke_crag_v2: 8/8; cached=True; requests=0


,id,type,action,gold_in_kept_titles,answer_contains_gold,n_llm_calls,cache_hit
0,5add1d575542992c1e3a2540,bridge,Correct,True,False,0,True
1,5ac55ea55542993e66e82377,bridge,Correct,True,False,0,True
2,5ac3a76e554299741d48a2be,bridge,Correct,True,False,0,True
3,5a7133565542994082a3e65c,bridge,Correct,True,False,0,True
4,5ac15de55542991316484afb,comparison,Incorrect,False,True,0,True
5,5a8cea40554299441c6b9f76,comparison,Correct,True,False,0,True
6,5a8ce15d554299653c1aa12f,comparison,Incorrect,False,False,0,True
7,5a7c634c55429907fabeef79,comparison,Ambiguous,False,False,0,True


For `5add1d575542992c1e3a2540`, the route was **Correct**. At least one annotated title contributed a kept strip: **True**. The literal gold substring test returned **False**. The answer was `I don't have enough information in the provided evidence to answer this question.` versus reference `Prussian`. Inspect the saved strips before interpreting this as success; this run made 0 model requests for the row.

For `5ac55ea55542993e66e82377`, the route was **Correct**. At least one annotated title contributed a kept strip: **True**. The literal gold substring test returned **False**. The answer was `Kurt Weill` versus reference `Kurt Julian Weill`. Inspect the saved strips before interpreting this as success; this run made 0 model requests for the row.

## 11. Evaluation on the first 50 slice questions

This small descriptive sample is neither a held-out tuning protocol nor a statistically robust benchmark. Do not calibrate thresholds on these rows and then present them as unbiased evaluation.

For each question, **gold_title_recall@k = number of distinct annotated titles retrieved / number of annotated titles**; report the mean across questions. Retrieving one of two titles gives 0.5, not 1.0. The action table groups questions by whether *any* gold title was present. This binary grouping must not be confused with title recall or complete evidence.

**Answer substring match** is a deliberately weak string proxy, not exact-match accuracy or factual correctness. `no` can match `not`; a sentence may mention the reference while denying it. Conversely `Kurt Weill` need not contain `Kurt Julian Weill` despite referring to the same person. We preserve the requested literal definition and label its limitations.

Mean model calls measures this invocation's HTTP attempts. A warm checkpoint run should report zero; it does not mean the original experiment was free. Cache keys include corpus, model, prompt-version and evaluation settings. Legacy caches are retained but not trusted by the repaired implementation.

### Run the 50-question slice

**Motivation.** Measure retrieval coverage, routing and answer proxies separately.

**Paper mapping.** A tutorial evaluation rather than the paper's datasets, models or reported metrics.

**Next cell.** Resume per-ID checkpoints, display aggregate metrics and the action contingency table.

**Failure signals.** A failed ID stops the cell after bounded retries; sanitized traceback files identify it and completed IDs remain reusable.

**Read the output.** Compare mean title recall with the action table; inspect cache counts before interpreting mean calls.

In [11]:
from src.pipeline import run_slice_50_benchmark
eval_rows, metrics, failure_gallery = run_slice_50_benchmark(records, k=3, allow_web=False)
eval_df = pd.DataFrame(eval_rows)
print(json.dumps(metrics, indent=2))
display(pd.crosstab(eval_df["gold_in_hits"], eval_df["action"], rownames=["Any gold title retrieved"]))
assert len(eval_rows) == 50

slice_50_crag_v2: 1/50; cached=True; requests=0


slice_50_crag_v2: 2/50; cached=True; requests=0


slice_50_crag_v2: 3/50; cached=True; requests=0


slice_50_crag_v2: 4/50; cached=True; requests=0


slice_50_crag_v2: 5/50; cached=True; requests=0


...

slice_50_crag_v2: 48/50; cached=True; requests=0


slice_50_crag_v2: 49/50; cached=True; requests=0


slice_50_crag_v2: 50/50; cached=True; requests=0


{
  "n_questions": 50,
  "gold_title_recall@k": 0.42,
  "answer_substring_match": 0.26,
  "mean_llm_calls": 0.0,
  "total_llm_calls": 0,
  "cached_rows": 50,
  "action_when_gold_present_vs_absent": {
    "Ambiguous": {
      "false": 1,
      "true": 0
    },
    "Correct": {
      "false": 3,
      "true": 30
    },
    "Incorrect": {
      "false": 12,
      "true": 4
    }
  }
}


action,Ambiguous,Correct,Incorrect
Any gold title retrieved,,,
False,1,3,12
True,0,30,4


## 12. Failure analysis

The gallery selects up to five substring nonmatches. They are candidate errors that require review. Aliases, abstentions, and incomplete references need human interpretation. Compare question → retrieved titles/scores → retained strips → generated answer. A relevant distractor may be a false positive, while a non-gold passage can still be useful evidence.

When all supplied passages fail to support either hop, Incorrect should have fired. A high-score distractor can instead force Correct because routing uses the maximum. Conversely a correct bridge paragraph can deserve relevance even before the second hop is present. Diagnose evaluator relevance separately from answer sufficiency; do not overwrite scores to force the expected story.

With web disabled, Incorrect cannot recover missing knowledge. This is an explicit limitation, not a reproduction of the paper's corrective web branch. Optional DDGS search uses real returned snippets and URLs only; network errors propagate, and snippets are not verified full-page evidence.

### Inspect candidate failures and distractors

**Motivation.** Turn aggregate numbers into specific repair hypotheses without inventing outcomes.

**Paper mapping.** Examines failure modes of the evaluator and correction policy.

**Next cell.** Show up to five nonmatches with all scores, then list high-scoring non-gold titles from evaluated rows.

**Failure signals.** Fewer than five nonmatches is reported honestly. Gold labels are annotations, not proof that every other paragraph is irrelevant.

**Read the output.** Decide whether each row reflects retrieval loss, false-positive relevance, refinement loss, unsupported generation, or metric mismatch.

In [12]:
print("Candidate nonmatches shown:", len(failure_gallery))
for row in failure_gallery:
    print("\nID:", row["id"], "Question:", row["question"])
    print("Action:", row["action"], "Answer:", row["generated_answer"], "Gold:", row["gold_answer"])
    display(pd.DataFrame(row["evaluations"]))
    print("Kept strips:", json.dumps(row["kept_strips"], ensure_ascii=False, indent=2))
suspects = []
for row in eval_rows:
    gold = set(records_by_id[row["id"]]["gold_titles"])
    for e in row["evaluations"]:
        if e["title"] not in gold and e["score"] >= .7:
            suspects.append({"id": row["id"], "action": row["action"], **e})
display(pd.DataFrame(suspects))
print("High-scoring non-gold passages:", len(suspects), "(inspect text before calling them false positives)")

Candidate nonmatches shown: 5

ID: 5add1d575542992c1e3a2540 Question: What nationality was Oliver Reed's character in the film Royal Flash?
Action: Correct Answer: I don't have enough information in the provided evidence to answer this question. Gold: Prussian


,title,score,label,why
0,Royal Flash (film),1.0,relevant,The evidence confirms Oliver Reed played Otto ...
1,Royal Flash,0.0,irrelevant,The evidence identifies the film and its sourc...
2,Harry Flashman,0.2,irrelevant,The evidence identifies the actor as Malcolm M...


Kept strips: [
  "Additionally, Oliver Reed appeared in the role of Otto von Bismarck, Alan Bates as Rudi von Sternberg, and Florinda Bolkan played Lola Montez."
]

ID: 5ac55ea55542993e66e82377 Question: Pacific Mozart Ensemble performed which German composer's Der Lindberghflug in 2002?
Action: Correct Answer: Kurt Weill Gold: Kurt Julian Weill


,title,score,label,why
0,The Flight Across the Ocean,0.85,relevant,The evidence explicitly identifies 'Der Lindbe...
1,Der Widerspänstigen Zähmung,0.00,irrelevant,The evidence discusses the opera 'Der Widerspä...
2,Pacific Mozart Ensemble,1.00,relevant,"The evidence explicitly states that in 2002, t..."


Kept strips: [
  "The Flight across the Ocean (\"Der Ozeanflug\") is a \"Lehrstück\" by the German dramatist Bertolt Brecht, inspired by \"We\", Charles Lindbergh's 1927 account of his transatlantic flight.",
  "Written for the Baden-Baden Music Festival, it was originally entitled \"Lindbergh's Flight\" (\"Der Lindberghflug\") and premiered in 1929 with music by Kurt Weill and Paul Hindemith in a broadcast by the Southwest German Radio Orchestra under the direction of Hermann Scherchen.",
  "Pacific Edge Voices (formerly The Pacific Mozart Ensemble (PME)) is a volunteer choral organization based in Berkeley, CA.",
  "A particular highlight came in 2002 when the chorus performed Kurt Weill’s Der Lindberghflug alongside works by Philip Glass, Meredith Monk and David Lang."
]

ID: 5ac3a76e554299741d48a2be Question: Who released the song "With or Without You" first, Jai McDowall or U2?
Action: Correct Answer: I don't have enough information in the provided evidence to answer this question

...

Kept strips: [
  "Lake Louisvilla is a neighborhood partially located in Louisville, Kentucky.",
  "It is located between Westport Road in Louisville and KY 22 in Oldham County."
]

ID: 5ac1a4745542991316484b82 Question: Para Hills West, South Australia lies within a city with what estimated population?
Action: Correct Answer: I don't have enough information in the provided evidence to answer this question. Gold: 138,535


,title,score,label,why
0,"Para Hills West, South Australia",1.0,relevant,The evidence identifies Para Hills West as bei...
1,"Gulfview Heights, South Australia",0.7,relevant,The question asks for the estimated population...
2,"Para Hills, South Australia",0.3,relevant,The evidence identifies Para Hills as a suburb...


Kept strips: [
  "Para Hills West is a suburb of Adelaide, South Australia, and is within the City of Salisbury.",
  "Gulfview Heights is a small suburb of Adelaide, South Australia and is within the City of Salisbury and City of Tea Tree Gully local government area."
]


,id,action,title,score,label,why
0,5ac55ea55542993e66e82377,Correct,The Flight Across the Ocean,0.85,relevant,The evidence explicitly identifies 'Der Lindbe...
1,5ac1a4745542991316484b82,Correct,"Gulfview Heights, South Australia",0.70,relevant,The question asks for the estimated population...
2,5ac55e835542993e66e82375,Correct,O.K. Corral hearing and aftermath,1.00,relevant,The question asks for the date of the event (g...
3,5a8d9e0f554299441c6ba015,Correct,Lock (film),0.90,relevant,The evidence identifies the specific 2016 Punj...
4,5ac39f7b554299218029dbe7,Correct,Skelly Oil,0.85,relevant,The evidence identifies the founders of Skelly...
5,5ae6363b55429929b0807af0,Correct,2010–11 Biathlon World Cup – Pursuit Men,1.00,relevant,Identifies the defending titlist (Martin Fourc...
6,5ab5ecd75542992aa134a3e6,Correct,Tien (TV channel),0.90,relevant,The evidence identifies John de Mol as the Dut...
7,5ade4da055429939a52fe878,Correct,Short films by Studio Ghibli,1.00,relevant,The evidence directly states that Studio Ghibl...
8,5ae7940c55429952e35ea984,Correct,Ane Mærsk Mc-Kinney Uggla,1.00,relevant,The evidence explicitly identifies Ane Uggla a...
9,5a7296815542991f9a20c515,Correct,ISSpresso,1.00,relevant,The evidence explicitly states that the first ...


High-scoring non-gold passages: 13 (inspect text before calling them false positives)


## Common errors and recovery

- **AGNESAI_API_KEY vs AGNES_API_KEY:** only the former is read. Confirm presence in Windows user environment settings, then restart the launcher. Never paste keys into cells, files, screenshots or logs.
- **HF download:** `HF_TOKEN` must reach the process. Use `hotpotqa/hotpot_qa`, `distractor`, `validation`; check network access and the `data/cache/hotpot_distractor_val` cache. Do not substitute invented records.
- **429 / transient server errors:** bounded exponential backoff retries the request. If exhausted, rerun the failed benchmark cell; successful per-ID and primitive caches remain. Authentication failures require correcting credentials, not endless retries.
- **JSON fences or extra prose:** the parser extracts a valid object and validates its schema. Invalid scores/IDs raise and are not cached. A retry may fix a transient formatting failure; persistent failures require inspecting the prompt/schema without exposing secrets.
- **Qdrant lock:** only one process/kernel may own `data/qdrant`. Run the closing cell, shut down the other kernel, or restart it. Do not delete the database or lock file while another kernel is active. Run quick-check and tutorial sequentially.
- **Imports:** launch from the repo or its notebooks folder using the registered `Python (CRAG Tutorial)` kernel. `run.cmd` installs dependencies into `.venv`; a different notebook kernel may not have them.

## Limits and next questions

This course uses lexical SVD vectors, an uncalibrated LLM evaluator, simple sentence boundaries, no default web recovery, and only 50 evaluation questions. It cannot establish paper-level accuracy, production readiness, or universal improvement over naive RAG.

Continue with [`02_naive_vs_crag_comparison.ipynb`](02_naive_vs_crag_comparison.ipynb), which runs the fixed naive baseline and CRAG over the identical 50 questions and top-3 passages, then manually reviews every answer pair. A separate tuning/evaluation split remains future work.

### Release the local database lock

**Motivation.** Allow the next notebook or fresh kernel to reopen the embedded store safely.

**Paper mapping.** Operational cleanup, independent of the CRAG algorithm.

**Next cell.** Close the shared Qdrant client after all inspection is complete.

**Failure signals.** If a previous cell crashed, run this cell manually or shut down its kernel before opening another notebook.

**Read the output.** The confirmation means this kernel released its client; another process can still own a separate lock.

In [13]:
close_qdrant_client()
print("Qdrant client closed.")

Qdrant client closed.
